# 01_service — Publish Actions over HTTP with FastApiAdapter

The same `Action` becomes an HTTP endpoint **without changing its code**. The adapter builds a FastAPI app: it validates the request, calls `auth_coordinator` to build a `Context`, runs the `Action`, and maps the `Result` back to JSON. OpenAPI is generated from `Params`/`Result` and `@meta`.

This notebook uses FastAPI's in-process `TestClient`, so it runs **without starting a server** — perfect for Colab. In production you serve the returned app with `uvicorn module:app`.

**What's new**

| Concept | Description |
|---------|-------------|
| `FastApiAdapter(machine, auth_coordinator, title=).post(...).build()` | Fluent builder → FastAPI app |
| `auth_coordinator` (required) | `None` → `TypeError`; open API uses `NoAuthCoordinator()` |
| OpenAPI from code | schema + `/docs` derived from `Params`/`Result`/`@meta` |
| `request_model`/`response_model` + `*_mapper` | bridge an external schema to the contract at the adapter |
| error mapping | `AuthorizationError` → 403, `ValidationFieldError` → 422, else → 500 |

> Install with the FastAPI extra (and `httpx`, which `TestClient` needs): `pip install "aoa-action-machine[fastapi]" httpx`.

In [ ]:
!pip install "aoa-action-machine[fastapi]" httpx

In [ ]:
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

from aoa.action_machine.adapters.fastapi import FastApiAdapter
from aoa.action_machine.auth import ApplicationRole, NoAuthCoordinator, GuestRole
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain, Role, Params, Result

In [ ]:
class GreetingDomain(BaseDomain):
    name = "greeting"
    description = "Greetings domain"


class AdminRole(ApplicationRole):
    name = "admin"
    description = "Administrator"


class GreetParams(BaseParams):
    name: str = Field(description="Person to greet")


class GreetResult(BaseResult):
    message: str = Field(description="Greeting message")

## Two operations — one open, one protected

`GreetAction` is open (`@check_roles(GuestRole)`); `AdminPingAction` requires `AdminRole`, so an anonymous caller is rejected with 403. Neither knows anything about HTTP.

In [ ]:
@meta(description="Greet a person", domain=GreetingDomain)
@check_roles(GuestRole)
class GreetAction(BaseAction[GreetParams, GreetResult]):

    @summary_aspect("Build greeting")
    async def greet_summary(self, params, state, box, connections):
        return GreetResult(message=f"Hello, {params.name}!")


@meta(description="Admin-only ping", domain=GreetingDomain)
@check_roles(AdminRole)
class AdminPingAction(BaseAction[GreetParams, GreetResult]):

    @summary_aspect("Admin ping")
    async def ping_summary(self, params, state, box, connections):
        return GreetResult(message=f"pong for {params.name}")

## External v2 schema, bridged at the adapter + `build_app`

`GreetV2Body`/`GreetV2Response` use different field names (`to`/`greeting`); `params_mapper`/`response_mapper` translate them to the operation's `Params`/`Result` — without touching `GreetAction`.

In [ ]:
class GreetV2Body(BaseModel):
    to: str = Field(description="Recipient (v2 field name)")


class GreetV2Response(BaseModel):
    greeting: str = Field(description="Greeting (v2 field name)")


def build_app():
    machine = ActionProductMachine()
    return (
        FastApiAdapter(machine=machine, auth_coordinator=NoAuthCoordinator(), title="Greetings API")
        .post("/greet", GreetAction, tags=["greetings"])
        .post("/admin/ping", AdminPingAction, tags=["admin"])
        .post(
            "/greet/v2",
            GreetAction,
            request_model=GreetV2Body,
            response_model=GreetV2Response,
            params_mapper=lambda body: GreetParams(name=body.to),
            response_mapper=lambda result: GreetV2Response(greeting=result.message),
            tags=["greetings"],
        )
        .build()
    )

## Run

`TestClient` drives the app in-process — no server, no `await`. The open endpoint returns 200, the protected one 403 for an anonymous caller, and `/greet/v2` bridges the external schema.

In [ ]:
def main() -> None:
    client = TestClient(build_app())

    r = client.get("/health")
    print(f"GET  /health     -> {r.status_code} {r.json()}")

    r = client.post("/greet", json={"name": "Alice"})
    print(f"POST /greet      -> {r.status_code} {r.json()}")

    r = client.post("/admin/ping", json={"name": "Alice"})
    print(f"POST /admin/ping -> {r.status_code} {r.json()}   (anonymous, AdminRole required)")

    r = client.post("/greet/v2", json={"to": "Bob"})
    print(f"POST /greet/v2   -> {r.status_code} {r.json()}   (external schema bridged)")


main()